<a href="https://colab.research.google.com/github/enslern/The-AI-engineer-course-2026-Complete-AI-bootcamp/blob/main/pinecone_vectordatabase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**installing the packages and loading the api key**

In [1]:
!pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 9.6 MB/s eta 0:00:00


In [2]:
import pinecone

In [3]:
from pinecone import Pinecone,ServerlessSpec

In [5]:
import os
import sys
sys.path.append(os.path.abspath("/content/drive/MyDrive/Colab Notebooks/resources"))

In [7]:
import config

In [17]:
os.environ["PINECONE_API_KEY"] = config.PINECONE_API_KEY

In [25]:
pc=Pinecone()

In [23]:
pc.list_indexes()

IndexList([<name='my-index', ready=True>])

##**creating and deleting pinecone index**

In [26]:
index_name="my-index"
dimension=3
metric="cosine"

In [30]:
if index_name in  [index.name for index in pc.list_indexes()]:
  pc.delete_index(index_name)
  print(f"{index_name} succesfully deleted!!")
else:
  print(f"{index_name} not in the index list")

my-index not in the index list


In [31]:
pc.create_index(name=index_name,
                dimension=dimension,
                metric=metric,
                spec=ServerlessSpec(
                    region="us-east-1",
                    cloud="aws"
                ))

Name:,my-index
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://my-index-toy62j7.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [32]:
pc.list_indexes()

IndexList([<name='my-index', ready=True>])

In [33]:
#create the assignment index
pc.create_index(
    name="my-index-2",
    dimension=1536,
    metric="euclidean",
    spec=ServerlessSpec(
        region="us-east-1",
        cloud="aws"
    )
)

Name:,my-index-2
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://my-index-2-toy62j7.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [34]:
pc.list_indexes()

IndexList([<name='my-index-2', ready=True>, <name='my-index', ready=True>])

##**upserting data to a pinecone vector database**

In [36]:
index=pc.Index(name=index_name)

In [40]:
index.upsert(
    vectors=[
        ("Dog", [4., 0., 1.]),
        ("Cat", [4., 0., 1.]),
        ("Chicken", [2., 2., 1.]),
        ("Elephant", [6., 2., 3.]),
        ("Mantis", [4., 0., 1.])
    ]
)

UpsertResponse(upserted_count=5)

##**Getting to know the fine web data set and loading it to colab**

In [43]:
import sys
# Clean up sys.path to ensure there are no function objects causing importlib to crash
sys.path = [path for path in sys.path if isinstance(path, str)]

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

In [55]:
fw=load_dataset("HuggingFaceFW/fineweb",name="sample-10BT",streaming=True,split="train")

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

In [56]:
fw

IterableDataset({
    features: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count'],
    num_shards: 15
})

##**upserting data from text file and using embedding algorithm**

In [46]:
model=SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [51]:
pc.create_index(name="text",
                dimension=model.get_sentence_embedding_dimension(),
                metric="cosine",
                spec=ServerlessSpec(
                    cloud="aws",
                    region="us-east-1"
                ))

/tmp/ipykernel_850/494096068.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension=model.get_sentence_embedding_dimension(),


Name:,text
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://text-toy62j7.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [61]:
# Explicitly use the 'text' index which has 384 dimensions matching your SentenceTransformer model
text_index = pc.Index(name="text")

subset_size = 1000
vectors_to_upsert = []
for i, item in enumerate(fw):
  if i >= subset_size:
    break
  text = item["text"]
  unique_id = str(item["id"])
  language = item["language"]
  embedding = model.encode(text, show_progress_bar=False).tolist()
  metadata = {"language": language}
  vectors_to_upsert.append((unique_id, embedding, metadata))
  if i % 100 == 0:
    print(f"embedded {i} items...")

batch_size = 100
for i in range(0, len(vectors_to_upsert), batch_size):
  batch = vectors_to_upsert[i:i+batch_size]
  text_index.upsert(vectors=batch)
  print(f"upserted {i}:{i+batch_size} vectors")
print("all vectors successfully upserted")

embedded 0 items...
embedded 100 items...
embedded 200 items...
embedded 300 items...
embedded 400 items...
embedded 500 items...
embedded 600 items...
embedded 700 items...
embedded 800 items...
embedded 900 items...
upserted 0:100 vectors
upserted 100:200 vectors
upserted 200:300 vectors
upserted 300:400 vectors
upserted 400:500 vectors
upserted 500:600 vectors
upserted 600:700 vectors
upserted 700:800 vectors
upserted 800:900 vectors
upserted 900:1000 vectors
all vectors successfully upserted
